<a href="https://colab.research.google.com/github/FranciscoBPereira/Salerno_October2026/blob/main/Salerno2026_Example2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install libraries to run this notebook.

!pip install -q transformers datasets accelerate evaluate scikit-learn


**1. Load Dataset**

In [ ]:
# Load Rotten Tomatoes dataset: https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes

from datasets import load_dataset
from transformers import set_seed


SEED = 42
set_seed(SEED)  # reproducibility (classifier-head initialisation, data shuffling, dropout)

# Prepare data and splits
# validation -> used during training; test -> used only once, for the final evaluation
tomatoes = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_data, val_data, test_data = tomatoes["train"], tomatoes["validation"], tomatoes["test"]


tomatoes

In [ ]:
# Print some reviews

for i in range(5):
    print(train_data[i])

**2. Pretrained Model Selection**

In [ ]:
from transformers import AutoTokenizer

# Model selection
# https://huggingface.co/google-bert/bert-base-uncased
# Pretrained model on English language using a masked language modeling (MLM) objective.

model_id = 'bert-base-uncased'

# Tokenization

tokenizer = AutoTokenizer.from_pretrained(model_id)

def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)


# Tokenize train/validation/test data
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_val = val_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)


In [ ]:
# Define training arguments

import numpy as np
import evaluate
from transformers import TrainingArguments, DataCollatorWithPadding

# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def make_training_args(output_dir, learning_rate, num_train_epochs):
    """Training arguments; each model gets its own output directory."""
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=learning_rate,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        seed=SEED,
        report_to='none',
    )


# Define evaluation metrics
f1_metric = evaluate.load('f1')
accuracy_metric = evaluate.load('accuracy')


def compute_metrics(eval_pred):
    """Calculate accuracy and F1 score (positive class = 1)"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_metric.compute(predictions=predictions, references=labels)['accuracy'],
        'f1': f1_metric.compute(predictions=predictions, references=labels)['f1'],
    }


def count_parameters(model):
    """Return (trainable, total) number of parameters"""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

**3. Model A: Add a classification head without finetuning the pretrained model**

In [ ]:

# Load ModelA - This model will just adjust its classification head

from transformers import AutoModelForSequenceClassification

set_seed(SEED)

modelA = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

# Freeze all weights from pretrained BERT module

for name, param in modelA.named_parameters():

     # Trainable classification head
     if name.startswith("classifier"):
        param.requires_grad = True

      # Freeze everything else
     else:
        param.requires_grad = False

trainable, total = count_parameters(modelA)
print(f'Model A - trainable parameters: {trainable:,} of {total:,}')

In [ ]:
from transformers import Trainer

# Only the linear head (768*2 + 2 parameters) is trained

L_rateA = 1e-3
EpochsA = 3

training_args_A = make_training_args('modelA', learning_rate = L_rateA, num_train_epochs = EpochsA)

# Trainer which executes the training process
trainerA = Trainer(
    model=modelA,
    args=training_args_A,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainerA.train()

# Final evaluation on the held-out test set
trainerA.evaluate(tokenized_test, metric_key_prefix='test')

**4. Model B: Finetuning the pretrained model**

In [ ]:
set_seed(SEED)

modelB = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

L_rateB = 2e-5
EpochsB = 3

training_args_B = make_training_args('modelB', learning_rate = L_rateB, num_train_epochs = EpochsB)

# Trainer which executes the training process
trainerB = Trainer(
    model=modelB,
    args=training_args_B,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainerB.train()

# Final evaluation on the held-out test set
trainerB.evaluate(tokenized_test, metric_key_prefix='test')


trainerB.evaluate()


In [ ]:
# ModelB number of parameters

trainable, total = count_parameters(modelB)
print(f'Model B - trainable parameters: {trainable:,} of {total:,}')

# Vocabulary size

print('Vocabulary Size: ', tokenizer.vocab_size)